# 🇻🇳 App Phân Loại Tin Tức Vietnamnet — Ensemble SVM + PhoBERT

| Cell | Nội dung |
|------|----------|
| **0. Kiểm tra** | Xác nhận thư viện và cả 2 mô hình |
| **1. Kiểm tra app** | Xác nhận `app_combined.py` tồn tại |
| **2. Khởi động** | Mở Streamlit tại `http://localhost:8501` |

> Sau khi train xong cả SVM lẫn PhoBERT, chạy **Cell 0 → Cell 2** là dùng được ngay.

In [1]:
# -- Kiểm tra môi trường ------------------------------------------------
import os, sys

SVM_PIPELINE  = os.path.normpath(os.path.join(os.getcwd(), '..', 'SVM',     'model', 'inference_pipeline.pkl'))
PHO_MODEL_DIR = os.path.normpath(os.path.join(os.getcwd(), '..', 'PhoBERT', 'model'))
PHO_CONFIG    = os.path.join(PHO_MODEL_DIR, 'label_config.json')
PHO_MODEL     = os.path.join(PHO_MODEL_DIR, 'model.safetensors')
PHO_THR       = os.path.join(PHO_MODEL_DIR, 'thresholds.json')

print('Kiểm tra thư viện:')
ok = True
missing_pkgs = []
for _pkg in ['streamlit', 'requests', 'bs4', 'pyvi', 'torch', 'transformers',
             'numpy', 'pandas', 'scipy', 'sklearn']:
    try:
        __import__(_pkg)
        print(f'  OK  {_pkg}')
    except ImportError:
        print(f'  MISS {_pkg}')
        missing_pkgs.append(_pkg)
        ok = False

print()
print('Kiểm tra model SVM:')
if os.path.exists(SVM_PIPELINE):
    print(f'  OK  inference_pipeline.pkl  ({os.path.getsize(SVM_PIPELINE)/1e6:.1f} MB)')
else:
    print(f'  MISS {SVM_PIPELINE}')
    ok = False

print()
print('Kiểm tra model PhoBERT:')
_required_pho = [
    ('model.safetensors',     PHO_MODEL),
    ('label_config.json',     PHO_CONFIG),
    ('config.json',           os.path.join(PHO_MODEL_DIR, 'config.json')),
    ('tokenizer_config.json', os.path.join(PHO_MODEL_DIR, 'tokenizer_config.json')),
]
for _name, _path in _required_pho:
    if os.path.exists(_path):
        print(f'  OK  {_name}  ({os.path.getsize(_path)/1e6:.1f} MB)')
    else:
        print(f'  MISS {_name}')
        ok = False

if os.path.exists(PHO_THR):
    print('  OK  thresholds.json -- Hiệu chỉnh ngưỡng: BẬt')
else:
    print('  CẢNH BÁO thresholds.json chưa có -- app dùng threshold mặc định')

if not ok:
    print()
    print('=' * 60)
    print('  KHÔNG THỂ TIẾP TỤC')
    print('=' * 60)
    if not os.path.exists(SVM_PIPELINE):
        print('  Model SVM chưa được tạo.')
        print('  -> Chạy main_SVM.ipynb (Section 7) để export inference_pipeline.pkl.')
    if not os.path.exists(PHO_MODEL):
        print('  Model PhoBERT chưa được tạo.')
        print('  -> Chạy main_PhoBERT.ipynb (Section 7) để export model.')
    if missing_pkgs:
        print('  Thiếu thư viện. Cài đặt:')
        print(f'     pip install {chr(32).join(missing_pkgs)}')
    print('=' * 60)
    raise SystemExit('Chua san sang -- xem huong dan o tren.')

print()
print('Sẵn sàng! Chạy cell tiếp theo để khởi động app.')

Kiểm tra thư viện:
  OK  streamlit
  OK  requests
  OK  bs4
  OK  pyvi
  OK  torch


C:\Users\DELL\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  OK  transformers
  OK  numpy
  OK  pandas
  OK  scipy
  OK  sklearn

Kiểm tra model SVM:
  OK  inference_pipeline.pkl  (29.7 MB)

Kiểm tra model PhoBERT:
  OK  model.safetensors  (540.1 MB)
  OK  label_config.json  (0.0 MB)
  OK  config.json  (0.0 MB)
  OK  tokenizer_config.json  (0.0 MB)
  OK  thresholds.json -- Hiệu chỉnh ngưỡng: BẬt

Sẵn sàng! Chạy cell tiếp theo để khởi động app.


---
## ✍️ Cell 1 — Kiểm tra App

Xác nhận `app_combined.py` tồn tại và các tính năng chính.

In [2]:
import os

_app_file = os.path.join(os.getcwd(), 'app_combined.py')

if os.path.exists(_app_file):
    _size = os.path.getsize(_app_file) / 1024
    print(f'✅ app_combined.py tồn tại  ({_size:.1f} KB)')

    with open(_app_file, encoding='utf-8') as f:
        _src = f.read()
    if 'ensemble_predict' in _src:
        print('✅ Dự đoán kết hợp: có trong app')
    if 'get_top_keywords' in _src:
        print('✅ Top keywords: có trong app')
    if 'batch' in _src.lower():
        print('✅ Nhiều URL: có trong app')
else:
    print(f'❌ Không tìm thấy: {_app_file}')
    print('   -> Kiểm tra lại thư mục Combined_Model_App/')

print('-> Chạy Cell 2 để khởi động app.')

✅ app_combined.py tồn tại  (30.5 KB)
✅ Dự đoán kết hợp: có trong app
✅ Top keywords: có trong app
✅ Nhiều URL: có trong app
-> Chạy Cell 2 để khởi động app.


---
## 🚀 Cell 2 — Khởi Động App

Mở **http://localhost:8501** trên trình duyệt.

> Cell này chạy cho đến khi bạn nhấn **■ Stop** (Interrupt Kernel).  
> Hoặc mở terminal riêng: `streamlit run app_combined.py`

In [ ]:
import subprocess, sys, os, threading, webbrowser, time

_cmd = [sys.executable, '-m', 'streamlit', 'run', 'app_combined.py',
        '--server.port', '8501', '--server.headless', 'true']
print('Chạy:', ' '.join(_cmd))
print('Mở trình duyệt: http://localhost:8501')
print('Nhấn ■ Interrupt Kernel để dừng.')
print()

threading.Thread(
    target=lambda: (time.sleep(4), webbrowser.open('http://localhost:8501')),
    daemon=True
).start()

subprocess.run(_cmd, cwd=os.getcwd())

Chạy: C:\Users\DELL\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m streamlit run app_combined.py --server.port 8501 --server.headless true
Mở trình duyệt: http://localhost:8501
Nhấn ■ Interrupt Kernel để dừng.

